In [2]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 

from sklearn.tree import DecisionTreeClassifier

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score

In [8]:

train = pd.read_csv("train.csv", usecols= ["Age", "Fare", "SibSp", "Parch", "Survived"])
train.dropna(inplace= True)
train.head()

,Survived,Age,SibSp,Parch,Fare
0,0,22.0,1,0,7.2500
1,1,38.0,1,0,71.2833
2,1,26.0,0,0,7.9250
3,1,35.0,1,0,53.1000
4,0,35.0,0,0,8.0500


In [9]:
train["family"] = train["SibSp"] + train["Parch"]
train.head()

,Survived,Age,SibSp,Parch,Fare,family
0,0,22.0,1,0,7.2500,1
1,1,38.0,1,0,71.2833,1
2,1,26.0,0,0,7.9250,0
3,1,35.0,1,0,53.1000,1
4,0,35.0,0,0,8.0500,0


In [10]:
train.drop(columns = ["SibSp", "Parch"], inplace= True) 
train.head()

,Survived,Age,Fare,family
0,0,22.0,7.2500,1
1,1,38.0,71.2833,1
2,1,26.0,7.9250,0
3,1,35.0,53.1000,1
4,0,35.0,8.0500,0


In [11]:
x = train.drop(columns=["Survived"])
y = train["Survived"]

In [12]:
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size= 0.2, 
    random_state=42
)

In [14]:
x_train.shape

(571, 3)

In [16]:
## Without Binarization

clf = DecisionTreeClassifier()

clf.fit(x_train, y_train)
y_pred = clf.predict(x_test)

In [19]:
accuracy_score(y_pred, y_test)

0.6153846153846154

In [20]:
np.mean(cross_val_score(
    DecisionTreeClassifier(), 
    x,
    y, 
    cv= 5, 
    scoring= "accuracy"
))

np.float64(0.6526445385600315)

In [21]:
x_train.head()

,Age,Fare,family
328,31.0,20.5250,2
73,26.0,14.4542,1
253,30.0,16.1000,1
719,33.0,7.7750,0
666,25.0,13.0000,0


In [41]:
# Now applying Binarizer

# I want to know if a person is travelling alone(0) or travelling with family(1)
# if family value is 0 than alone(0) but if more than 0 than with family(1)

from sklearn.preprocessing import Binarizer

trf = ColumnTransformer([
    ("bin", Binarizer(copy= True, threshold= 0), ['family'])
], remainder= "passthrough")

In [42]:
x_train_trf = trf.fit_transform(x_train)
x_test_trf = trf.transform(x_test)

In [44]:
DF = pd.DataFrame(x_train_trf, columns= [ "Bin_Family", "Age", "Fare"])
DF

,Bin_Family,Age,Fare
0,1.0,31.0,20.5250
1,1.0,26.0,14.4542
2,1.0,30.0,16.1000
3,0.0,33.0,7.7750
4,0.0,25.0,13.0000
...,...,...,...
566,1.0,46.0,61.1750
567,0.0,25.0,13.0000
568,0.0,41.0,134.5000
569,1.0,33.0,20.5250


In [45]:
clf2 = DecisionTreeClassifier()

clf2.fit(x_train_trf, y_train)
y_pred_2 = clf2.predict(x_test_trf)

accuracy_score(y_pred_2, y_test)

0.6363636363636364

In [46]:
x_trf = trf.fit_transform(x)

np.mean(cross_val_score(
    DecisionTreeClassifier(), 
    x_trf, 
    y, 
    cv= 10, 
    scoring= "accuracy"
))

np.float64(0.6304381846635367)